# Nigeria Energy Access & GDP Analysis
## Notebook 1: Data Cleaning & Preparation

**Project:** Powering Growth — Analysing the Relationship Between Energy Access and Economic Output in Nigeria (2000–2022)  
**Author:** [Your Name]  
**Date:** April 2026  
**Tools:** Python, Pandas, NumPy

---

### Objective
This notebook loads, inspects, cleans, and merges the raw datasets needed for the analysis.  
All cleaned outputs are saved to `../data/cleaned/` for use in subsequent notebooks.

### Datasets Used
| File | Source | Description |
|------|--------|-------------|
| `API_EG.ELC.ACCS.ZS_DS2_en_csv_v2.csv` | World Bank | Electricity access (% of population) |
| `API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv` | World Bank | GDP (current US$) |
| `API_NY.GDP.PCAP.CD_DS2_en_csv_v2.csv` | World Bank | GDP per capita (current US$) |
| `API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv` | World Bank | Energy use per capita (kg of oil equivalent) |

> **How to download:** Go to data.worldbank.org → Search each indicator → Select Nigeria + comparison countries → Download CSV

---
## Section 1: Import Libraries

In [1]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# ── Display Settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.float_format', '{:,.2f}'.format)  # clean number format

print('✅ Libraries loaded successfully')
print(f'   Pandas version  : {pd.__version__}')
print(f'   NumPy version   : {np.__version__}')

✅ Libraries loaded successfully
   Pandas version  : 3.0.2
   NumPy version   : 2.4.4


---
## Section 2: Set File Paths

In [2]:
# ── Directory Paths ───────────────────────────────────────────────────────────
RAW_DIR     = '../data/raw/'
CLEANED_DIR = '../data/cleaned/'

# Create cleaned folder if it doesn't exist yet
os.makedirs(CLEANED_DIR, exist_ok=True)

# ── Raw File Names ────────────────────────────────────────────────────────────
# UPDATE THESE to match the exact filenames you downloaded from the World Bank
FILE_ELEC_ACCESS  = 'API_EG.ELC.ACCS.ZS_DS2_en_csv_v2.csv'     # Electricity access %
FILE_GDP          = 'API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv'      # GDP current USD
FILE_GDP_PERCAP   = 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2.csv'      # GDP per capita
FILE_ENERGY_USE   = 'API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv'  # Energy use per capita

# ── Countries of Interest ─────────────────────────────────────────────────────
# Nigeria is the focus; others are for peer comparison in later notebooks
COUNTRIES = ['Nigeria', 'Ghana', 'Kenya', 'South Africa']

# ── Year Range ────────────────────────────────────────────────────────────────
START_YEAR = 2000
END_YEAR   = 2022

print(f'✅ Paths configured')
print(f'   Raw data folder     : {RAW_DIR}')
print(f'   Cleaned data folder : {CLEANED_DIR}')
print(f'   Year range          : {START_YEAR} – {END_YEAR}')
print(f'   Countries           : {COUNTRIES}')

✅ Paths configured
   Raw data folder     : ../data/raw/
   Cleaned data folder : ../data/cleaned/
   Year range          : 2000 – 2022
   Countries           : ['Nigeria', 'Ghana', 'Kenya', 'South Africa']


---
## Section 3: Helper Function — Load World Bank CSV

World Bank CSV files have a non-standard format: the first 4 rows are metadata headers.  
This function handles that automatically for every file we load.

In [3]:
def load_worldbank_csv(filepath, value_name):
    """
    Loads a World Bank indicator CSV and reshapes it from wide to long format.
    Handles the format: Series Name | Series Code | Country Name | Country Code | 2000 [YR2000] ...
    """

    # Step 1: Read the file directly — header is at row 0
    df_raw = pd.read_csv(filepath)

    # Step 2: Clean column names (strip whitespace)
    df_raw.columns = df_raw.columns.str.strip()

    # Step 3: Identify year columns — they look like '2000 [YR2000]'
    # Extract only columns that contain a year in our range
    all_cols = df_raw.columns.tolist()

    year_col_map = {}  # maps clean year '2000' -> actual column '2000 [YR2000]'
    for col in all_cols:
        for y in range(START_YEAR, END_YEAR + 1):
            if col.startswith(str(y)):
                year_col_map[str(y)] = col
                break

    print(f"  📅 Found {len(year_col_map)} year columns in: {filepath.split(chr(92))[-1]}")

    # Step 4: Keep only Country Name, Country Code, and year columns
    id_cols   = ['Country Name', 'Country Code']
    year_cols_actual = [year_col_map[y] for y in sorted(year_col_map.keys())]

    df = df_raw[id_cols + year_cols_actual].copy()

    # Step 5: Filter to countries of interest
    df = df[df['Country Name'].isin(COUNTRIES)].copy()

    # Step 6: Rename year columns to clean years '2000', '2001' etc.
    rename_map = {v: k for k, v in year_col_map.items()}
    df.rename(columns=rename_map, inplace=True)

    # Step 7: Drop rows where Country Name is NaN
    df.dropna(subset=['Country Name'], inplace=True)

    # Step 8: Reshape from wide to long format
    year_cols_clean = sorted(year_col_map.keys())
    df = df.melt(
        id_vars    = id_cols,
        value_vars = year_cols_clean,
        var_name   = 'year',
        value_name = value_name
    )

    # Step 9: Rename and cast types
    df.rename(columns={
        'Country Name' : 'country_name',
        'Country Code' : 'country_code'
    }, inplace=True)

    df['year'] = df['year'].astype(int)

    # Step 10: Sort for readability
    df.sort_values(['country_name', 'year'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df


print('✅ Helper function updated: load_worldbank_csv()')

✅ Helper function updated: load_worldbank_csv()


---
## Section 4: Load All Raw Datasets

In [4]:
# ── Load each dataset ─────────────────────────────────────────────────────────
print('Loading datasets...\n')

df_elec   = load_worldbank_csv(RAW_DIR + FILE_ELEC_ACCESS, 'electricity_access_pct')
print(f'  ✅ Electricity Access  — {df_elec.shape[0]} rows, {df_elec.shape[1]} columns')

df_gdp    = load_worldbank_csv(RAW_DIR + FILE_GDP, 'gdp_usd')
print(f'  ✅ GDP (Current USD)   — {df_gdp.shape[0]} rows, {df_gdp.shape[1]} columns')

df_gdppc  = load_worldbank_csv(RAW_DIR + FILE_GDP_PERCAP, 'gdp_per_capita_usd')
print(f'  ✅ GDP Per Capita      — {df_gdppc.shape[0]} rows, {df_gdppc.shape[1]} columns')

df_energy = load_worldbank_csv(RAW_DIR + FILE_ENERGY_USE, 'energy_use_per_capita')
print(f'  ✅ Energy Use/Capita   — {df_energy.shape[0]} rows, {df_energy.shape[1]} columns')

Loading datasets...

  📅 Found 23 year columns in: ../data/raw/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2.csv
  ✅ Electricity Access  — 92 rows, 4 columns
  📅 Found 23 year columns in: ../data/raw/API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv
  ✅ GDP (Current USD)   — 92 rows, 4 columns
  📅 Found 23 year columns in: ../data/raw/API_NY.GDP.PCAP.CD_DS2_en_csv_v2.csv
  ✅ GDP Per Capita      — 92 rows, 4 columns
  📅 Found 23 year columns in: ../data/raw/API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv
  ✅ Energy Use/Capita   — 92 rows, 4 columns


In [5]:
# DIAGNOSTIC: See what country names actually exist in your CSV files
import pandas as pd

# Read the first file directly
df_check = pd.read_csv('../data/raw/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2.csv', 
                        header=None, nrows=10)

print("=== FIRST 10 ROWS OF RAW FILE ===")
print(df_check.to_string())
print()

# Also check column names after skipping row 0
df_check2 = pd.read_csv('../data/raw/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2.csv', 
                         skiprows=0)
print("=== COLUMN NAMES ===")
print(df_check2.columns.tolist()[:10])
print()
print("=== FIRST 5 VALUES IN COLUMN 0 ===")
print(df_check2.iloc[:5, 0].tolist())

=== FIRST 10 ROWS OF RAW FILE ===
                                                 0               1             2             3              4              5              6              7              8              9              10             11             12             13             14             15             16             17             18             19             20             21             22             23             24             25             26
0                                       Series Name     Series Code  Country Name  Country Code  2000 [YR2000]  2001 [YR2001]  2002 [YR2002]  2003 [YR2003]  2004 [YR2004]  2005 [YR2005]  2006 [YR2006]  2007 [YR2007]  2008 [YR2008]  2009 [YR2009]  2010 [YR2010]  2011 [YR2011]  2012 [YR2012]  2013 [YR2013]  2014 [YR2014]  2015 [YR2015]  2016 [YR2016]  2017 [YR2017]  2018 [YR2018]  2019 [YR2019]  2020 [YR2020]  2021 [YR2021]  2022 [YR2022]
1           Access to electricity (% of population)  EG.ELC.ACCS.ZS         Ghana   

---
## Section 5: Inspect Each Dataset

Always inspect your raw data before touching it.  
This is standard practice in any professional analytics engagement.

In [5]:
def inspect_df(df, label):
    """
    Prints a structured summary of a DataFrame:
    shape, dtypes, missing values, and a sample of the data.
    """
    print('=' * 60)
    print(f'DATASET: {label}')
    print('=' * 60)
    print(f'  Shape          : {df.shape[0]} rows × {df.shape[1]} columns')
    print(f'  Year range     : {df["year"].min()} – {df["year"].max()}')
    print(f'  Countries      : {df["country_name"].unique().tolist()}')
    print()
    print('  Column Info:')
    print(df.dtypes.to_string())
    print()
    print('  Missing Values per Column:')
    print(df.isnull().sum().to_string())
    print()
    print('  Sample (first 5 rows):')
    print(df.head())
    print()


inspect_df(df_elec,   'Electricity Access (%)')
inspect_df(df_gdp,    'GDP (Current USD)')
inspect_df(df_gdppc,  'GDP Per Capita (USD)')
inspect_df(df_energy, 'Energy Use Per Capita')

DATASET: Electricity Access (%)
  Shape          : 92 rows × 4 columns
  Year range     : 2000 – 2022
  Countries      : ['Ghana', 'Kenya', 'Nigeria', 'South Africa']

  Column Info:
country_name                  str
country_code                  str
year                        int64
electricity_access_pct    float64

  Missing Values per Column:
country_name              0
country_code              0
year                      0
electricity_access_pct    0

  Sample (first 5 rows):
  country_name country_code  year  electricity_access_pct
0        Ghana          GHA  2000                   43.70
1        Ghana          GHA  2001                   44.80
2        Ghana          GHA  2002                   46.80
3        Ghana          GHA  2003                   48.30
4        Ghana          GHA  2004                   50.80

DATASET: GDP (Current USD)
  Shape          : 92 rows × 4 columns
  Year range     : 2000 – 2022
  Countries      : ['Ghana', 'Kenya', 'Nigeria', 'South Africa']

 

---
## Section 6: Handle Missing Values

World Bank data often has gaps — especially in early years for African economies.  
We document every decision we make here. In consulting, you must be able to defend your choices.

In [6]:
def summarise_missing(df, label):
    """Shows the count and percentage of missing values per country."""
    value_col = [c for c in df.columns if c not in ['country_name', 'country_code', 'year']][0]
    missing = (
        df.groupby('country_name')[value_col]
          .apply(lambda x: x.isnull().sum())
          .reset_index()
    )
    missing.columns = ['country_name', 'missing_count']
    missing['total_years'] = END_YEAR - START_YEAR + 1
    missing['missing_pct'] = (missing['missing_count'] / missing['total_years'] * 100).round(1)
    print(f'\n── Missing Values: {label} ──')
    print(missing.to_string(index=False))


summarise_missing(df_elec,   'Electricity Access')
summarise_missing(df_gdp,    'GDP')
summarise_missing(df_gdppc,  'GDP Per Capita')
summarise_missing(df_energy, 'Energy Use')


── Missing Values: Electricity Access ──
country_name  missing_count  total_years  missing_pct
       Ghana              0           23         0.00
       Kenya              0           23         0.00
     Nigeria              0           23         0.00
South Africa              0           23         0.00

── Missing Values: GDP ──
country_name  missing_count  total_years  missing_pct
       Ghana              0           23         0.00
       Kenya              0           23         0.00
     Nigeria              0           23         0.00
South Africa              0           23         0.00

── Missing Values: GDP Per Capita ──
country_name  missing_count  total_years  missing_pct
       Ghana              0           23         0.00
       Kenya              0           23         0.00
     Nigeria              0           23         0.00
South Africa              0           23         0.00

── Missing Values: Energy Use ──
country_name  missing_count  total_years  missing

In [7]:
def fill_missing_values(df, value_col):
    """
    Fills missing values per country using linear interpolation.

    Strategy (documented for consulting transparency):
    - Linear interpolation fills internal gaps between known data points.
    - Forward-fill handles any remaining gaps at the end of the series.
    - Back-fill handles gaps at the beginning of the series.
    - Rows still missing after all three steps are flagged.
    """
    df = df.copy()

    df[value_col] = (
        df.groupby('country_name')[value_col]
          .transform(lambda x: x.interpolate(method='linear')
                                 .ffill()
                                 .bfill())
    )

    remaining_missing = df[value_col].isnull().sum()
    if remaining_missing > 0:
        print(f'  ⚠️  {remaining_missing} values still missing in {value_col} — review manually')
    else:
        print(f'  ✅ {value_col} — all missing values resolved')

    return df


print('Filling missing values...\n')
df_elec   = fill_missing_values(df_elec,   'electricity_access_pct')
df_gdp    = fill_missing_values(df_gdp,    'gdp_usd')
df_gdppc  = fill_missing_values(df_gdppc,  'gdp_per_capita_usd')
df_energy = fill_missing_values(df_energy, 'energy_use_per_capita')

Filling missing values...

  ✅ electricity_access_pct — all missing values resolved
  ✅ gdp_usd — all missing values resolved
  ✅ gdp_per_capita_usd — all missing values resolved
  ✅ energy_use_per_capita — all missing values resolved


---
## Section 7: Validate Data Ranges

Check that values are within logical bounds.  
Electricity access must be 0–100%. GDP must be positive. Catching these early prevents misleading charts.

In [8]:
def validate_range(df, col, min_val=None, max_val=None, label=''):
    """
    Checks whether all values in a column fall within an expected range.
    Prints a warning and displays offending rows if any violations are found.
    """
    issues = pd.Series([False] * len(df))

    if min_val is not None:
        issues = issues | (df[col] < min_val)
    if max_val is not None:
        issues = issues | (df[col] > max_val)

    if issues.any():
        print(f'  ⚠️  {label}: {issues.sum()} value(s) outside expected range [{min_val}, {max_val}]')
        print(df[issues][['country_name', 'year', col]])
    else:
        print(f'  ✅ {label}: all values within expected range')


print('Validating data ranges...\n')
validate_range(df_elec,   'electricity_access_pct', min_val=0,  max_val=100, label='Electricity Access %')
validate_range(df_gdp,    'gdp_usd',                min_val=0,               label='GDP USD')
validate_range(df_gdppc,  'gdp_per_capita_usd',     min_val=0,               label='GDP Per Capita')
validate_range(df_energy, 'energy_use_per_capita',  min_val=0,               label='Energy Use Per Capita')

Validating data ranges...

  ✅ Electricity Access %: all values within expected range
  ✅ GDP USD: all values within expected range
  ✅ GDP Per Capita: all values within expected range
  ✅ Energy Use Per Capita: all values within expected range


---
## Section 8: Add Derived Columns

Calculated columns that will be useful in later analysis and visualisation.

In [9]:
# ── GDP in Billions (easier to read on charts) ─────────────────────────────
df_gdp['gdp_billion_usd'] = df_gdp['gdp_usd'] / 1e9

# ── Year-on-Year GDP Growth Rate (%) ──────────────────────────────────────
df_gdp['gdp_growth_rate_pct'] = (
    df_gdp.groupby('country_name')['gdp_usd']
          .pct_change() * 100
)

# ── Electricity Access Gap from 100% (how far from universal access) ───────
df_elec['access_gap_pct'] = 100 - df_elec['electricity_access_pct']

# ── Flag: Nigeria only (used to highlight Nigeria in comparison charts) ─────
for df in [df_elec, df_gdp, df_gdppc, df_energy]:
    df['is_nigeria'] = df['country_name'] == 'Nigeria'

print('✅ Derived columns added:')
print('   gdp_billion_usd      — GDP in billions (df_gdp)')
print('   gdp_growth_rate_pct  — Year-on-year GDP growth (df_gdp)')
print('   access_gap_pct       — Distance from universal electricity access (df_elec)')
print('   is_nigeria           — Boolean flag for Nigeria rows (all dataframes)')

# Preview
print('\nSample — GDP with derived columns (Nigeria):')
df_gdp[df_gdp['country_name'] == 'Nigeria'][['year', 'gdp_usd', 'gdp_billion_usd', 'gdp_growth_rate_pct']].head(8)

✅ Derived columns added:
   gdp_billion_usd      — GDP in billions (df_gdp)
   gdp_growth_rate_pct  — Year-on-year GDP growth (df_gdp)
   access_gap_pct       — Distance from universal electricity access (df_elec)
   is_nigeria           — Boolean flag for Nigeria rows (all dataframes)

Sample — GDP with derived columns (Nigeria):


,year,gdp_usd,gdp_billion_usd,gdp_growth_rate_pct
46,2000,"69,171,451,627.25",69.17,NaN
47,2001,"73,557,840,064.49",73.56,6.34
48,2002,"95,054,059,302.70",95.05,29.22
49,2003,"104,738,954,264.23",104.74,10.19
50,2004,"135,764,731,645.61",135.76,29.62
51,2005,"175,670,569,969.35",175.67,29.39
52,2006,"238,454,997,161.48",238.45,35.74
53,2007,"278,260,846,800.10",278.26,16.69


---
## Section 9: Merge All Datasets Into One Master Table

In [10]:
# ── Merge on country_name + country_code + year ───────────────────────────
merge_keys = ['country_name', 'country_code', 'year']

df_master = (
    df_elec[merge_keys + ['electricity_access_pct', 'access_gap_pct', 'is_nigeria']]
    .merge(df_gdp[merge_keys + ['gdp_usd', 'gdp_billion_usd', 'gdp_growth_rate_pct']],
           on=merge_keys, how='outer')
    .merge(df_gdppc[merge_keys + ['gdp_per_capita_usd']],
           on=merge_keys, how='outer')
    .merge(df_energy[merge_keys + ['energy_use_per_capita']],
           on=merge_keys, how='outer')
)

# ── Sort ──────────────────────────────────────────────────────────────────
df_master.sort_values(['country_name', 'year'], inplace=True)
df_master.reset_index(drop=True, inplace=True)

print(f'✅ Master dataset created')
print(f'   Shape    : {df_master.shape[0]} rows × {df_master.shape[1]} columns')
print(f'   Countries: {df_master["country_name"].unique().tolist()}')
print(f'   Years    : {df_master["year"].min()} – {df_master["year"].max()}')
print()
df_master.head(10)

✅ Master dataset created
   Shape    : 92 rows × 11 columns
   Countries: ['Ghana', 'Kenya', 'Nigeria', 'South Africa']
   Years    : 2000 – 2022



,country_name,country_code,year,electricity_access_pct,access_gap_pct,is_nigeria,gdp_usd,gdp_billion_usd,gdp_growth_rate_pct,gdp_per_capita_usd,energy_use_per_capita
0,Ghana,GHA,2000,43.70,56.30,False,"4,982,850,662.21",4.98,NaN,253.75,322.13
1,Ghana,GHA,2001,44.80,55.20,False,"5,314,872,854.44",5.31,6.66,263.54,313.17
2,Ghana,GHA,2002,46.80,53.20,False,"6,166,197,847.85",6.17,16.02,297.46,303.80
3,Ghana,GHA,2003,48.30,51.70,False,"7,632,723,555.66",7.63,23.78,358.40,281.22
4,Ghana,GHA,2004,50.80,49.20,False,"8,881,417,906.71",8.88,16.36,406.13,270.19
5,Ghana,GHA,2005,41.30,58.70,False,"10,744,568,381.45",10.74,20.98,478.61,261.89
6,Ghana,GHA,2006,55.10,44.90,False,"20,885,037,596.70",20.89,94.38,906.44,274.13
7,Ghana,GHA,2007,56.90,43.10,False,"24,827,339,138.49",24.83,18.88,"1,050.12",263.03
8,Ghana,GHA,2008,60.50,39.50,False,"28,679,383,241.07",28.68,15.52,"1,182.66",276.02
9,Ghana,GHA,2009,61.10,38.90,False,"26,048,720,005.52",26.05,-9.17,"1,047.70",274.27


---
## Section 10: Final Quality Check

In [11]:
print('FINAL QUALITY REPORT')
print('=' * 60)
print(f'  Total rows          : {len(df_master)}')
print(f'  Total columns       : {df_master.shape[1]}')
print(f'  Countries           : {df_master["country_name"].nunique()}')
print(f'  Year span           : {df_master["year"].min()} – {df_master["year"].max()}')
print()
print('  Missing Values in Master Dataset:')
print(df_master.isnull().sum().to_string())
print()
print('  Summary Statistics (Nigeria only):')
nigeria_cols = ['electricity_access_pct', 'gdp_billion_usd', 'gdp_growth_rate_pct', 'gdp_per_capita_usd']
print(df_master[df_master['country_name'] == 'Nigeria'][nigeria_cols].describe().round(2))

FINAL QUALITY REPORT
  Total rows          : 92
  Total columns       : 11
  Countries           : 4
  Year span           : 2000 – 2022

  Missing Values in Master Dataset:
country_name              0
country_code              0
year                      0
electricity_access_pct    0
access_gap_pct            0
is_nigeria                0
gdp_usd                   0
gdp_billion_usd           0
gdp_growth_rate_pct       4
gdp_per_capita_usd        0
energy_use_per_capita     0

  Summary Statistics (Nigeria only):
       electricity_access_pct  gdp_billion_usd  gdp_growth_rate_pct  \
count                   23.00            23.00                22.00   
mean                    51.96           363.61                12.13   
std                      5.09           192.76                18.47   
min                     43.20            69.17               -17.93   
25%                     47.80           207.06                 2.87   
50%                     52.50           375.75        

---
## Section 11: Save Cleaned Datasets

In [12]:
# ── Save master dataset (all countries) ───────────────────────────────────
master_path = CLEANED_DIR + 'master_energy_gdp.csv'
df_master.to_csv(master_path, index=False)
print(f'✅ Saved: {master_path}')

# ── Save Nigeria-only slice (used most in notebooks 02 and 03) ─────────────
nigeria_path = CLEANED_DIR + 'nigeria_energy_gdp.csv'
df_master[df_master['country_name'] == 'Nigeria'].to_csv(nigeria_path, index=False)
print(f'✅ Saved: {nigeria_path}')

# ── Confirm files exist ────────────────────────────────────────────────────
print()
print('Files in cleaned folder:')
for f in os.listdir(CLEANED_DIR):
    size_kb = os.path.getsize(CLEANED_DIR + f) / 1024
    print(f'   {f:45s}  {size_kb:.1f} KB')

✅ Saved: ../data/cleaned/master_energy_gdp.csv
✅ Saved: ../data/cleaned/nigeria_energy_gdp.csv

Files in cleaned folder:
   master_energy_gdp.csv                          11.3 KB
   nigeria_energy_gdp.csv                         2.8 KB


---
## Summary

| Step | Action | Status |
|------|--------|--------|
| 1 | Loaded 4 World Bank datasets | ✅ |
| 2 | Reshaped from wide to long format | ✅ |
| 3 | Filtered to 4 countries (2000–2022) | ✅ |
| 4 | Inspected shape, types, and missing values | ✅ |
| 5 | Filled missing values via interpolation | ✅ |
| 6 | Validated all columns against logical bounds | ✅ |
| 7 | Created derived columns (growth rate, GDP billions, access gap) | ✅ |
| 8 | Merged all datasets into one master table | ✅ |
| 9 | Saved `master_energy_gdp.csv` and `nigeria_energy_gdp.csv` | ✅ |

---

**Next:** Open `02_exploratory_analysis.ipynb` to begin answering the five analytical questions.